# ClusterGCN sa lokalnim reindeksiranjem i int16 strukturom

Modifikacija ClusterGCN pristupa: umesto globalnih int64 indeksa nad celim grafom,
svaki klaster dobija lokalnu numeraciju od nule, pa lokalni indeksi staju u int16
(opseg do 32.767). Struktura grafa se skladisti i prenosi na karticu u int16, a
neposredno pre prolaska kroz mrezu podize se na najuzi tip koji kerneli prihvataju,
uz dodavanje ofseta klastera.

Tri varijante pri istom K i istom protokolu treninga:

| Varijanta | Potezi izmedju klastera u batch-u | Skladistenje | Napomena |
|---|---|---|---|
| osnova | zadrzani | int64 | postojeci ClusterLoader, identicno Koraku 2 |
| izolovano_int64 | odbaceni | int64 | izolovani klasteri, kontrola za int16 |
| izolovano_int16 | odbaceni | int16 | predlozena modifikacija |

Poredjenje osnove i izolovano_int64 meri koliko kosta izolacija klastera. Varijante
izolovano_int64 i izolovano_int16 moraju imati identicnu sekvencu batch-eva, sto se
proverava kontrolnim zbirovima, pa se smeju razlikovati samo skladiste, prenos i memorija,
dok razlika tacnosti sme biti samo od GPU nedeterminizma.

Protokol identican Koraku 2: GraphSAGE, L = 3, skriveni sloj 256, koeficijent ucenja 0,01,
5.000 gradijentnih koraka, 20 klastera po batch-u, ocena na svakih 250 koraka.

Napomena za vreme: osnova koristi ClusterLoader sa 8 radnih procesa, a izolovane varijante
sastavljaju batch u glavnom procesu, pa je vreme po koraku uporedivo samo izmedju dve
izolovane varijante.

Drugi deo meri skladiste i prenos strukture na sva cetiri grafa iz izvestaja, bez treninga.

Procena trajanja: oko tri sata za prvi deo (90 treninga) i do sat vremena za drugi deo,
METIS na products-u i Reddit-u je spor. Sve ide u jednoj sesiji. Po potrebi se lista
SEEDOVI ili KLASTERI skracuje u prvoj celiji, merenje preskace vec upisane redove.


In [ ]:
SEEDOVI = [42, 123, 7, 1, 2, 3, 4, 5, 6, 8]
KLASTERI = [100, 500, 2000]
KLASTERA_PO_BATCHU = 20
VARIJANTE = ['osnova', 'izolovano_int64', 'izolovano_int16']


In [ ]:
import os, sys, time, json, gc, csv, random, shutil, logging
from pathlib import Path
from datetime import datetime

import torch
import torch.nn.functional as F
import numpy as np

from torch_geometric.nn import SAGEConv
from torch_geometric.loader import ClusterData, ClusterLoader
from torch_geometric.utils import to_undirected
from ogb.nodeproppred import PygNodePropPredDataset, Evaluator
import torch_geometric.data

torch.serialization.add_safe_globals([
    torch_geometric.data.data.DataEdgeAttr,
    torch_geometric.data.data.DataTensorAttr,
    torch_geometric.data.storage.GlobalStorage,
])

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = torch.device('cuda')

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s',
                    handlers=[logging.StreamHandler(sys.stdout)], force=True)
log = logging.getLogger(__name__)

cc = torch.cuda.get_device_capability(0)
log.info(f'kartica {torch.cuda.get_device_name(0)}, compute capability {cc[0]}.{cc[1]}')

REZ = Path('rezultati_int16')
REZ.mkdir(exist_ok=True)


def postavi_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


def direktorijum(seed):
    p = REZ / f'run_{seed}'
    p.mkdir(parents=True, exist_ok=True)
    return p


def upisi_red(putanja, red):
    novi = not putanja.exists()
    with open(putanja, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(red.keys()))
        if novi:
            w.writeheader()
        w.writerow(red)


def vec_uradjeno(putanja, kolona):
    if not putanja.exists():
        return set()
    with open(putanja) as f:
        return {r[kolona] for r in csv.DictReader(f)}


## Tipovi indeksa na ovoj verziji PyTorch-a

Lokalna provera na torch 2.6.0 je pokazala da gather i scatter primaju samo int64,
index_select prima i int32, a int16 ne prima nijedna operacija. Ova celija to potvrdjuje
na verziji sa platforme i bira najuzi tip sa kojim SAGEConv radi i unapred i unazad.


In [ ]:
INT16_MAKS = 32767

x_proba = torch.randn(8, 4, device=device)
idx_proba = torch.tensor([0, 3, 7], device=device)
ei_proba = torch.tensor([[0, 1, 2, 5], [1, 2, 0, 3]], device=device)
conv_proba = SAGEConv(4, 4).to(device)


def radi(fn):
    try:
        fn()
        return True
    except (RuntimeError, IndexError, TypeError, NotImplementedError):
        return False


def sageconv_radi(tip):
    def f():
        izlaz = conv_proba(x_proba, ei_proba.to(tip))
        izlaz.sum().backward()
    return radi(f)


log.info(f'torch {torch.__version__}')
RACUNSKI_TIP = None
for tip in [torch.int16, torch.int32, torch.int64]:
    st = {
        'index_select': radi(lambda: torch.index_select(x_proba, 0, idx_proba.to(tip))),
        'gather': radi(lambda: torch.gather(x_proba, 0, torch.zeros(2, 4, dtype=tip, device=device))),
        'scatter_add': radi(lambda: torch.zeros(8, 4, device=device).scatter_add_(
            0, torch.zeros(2, 4, dtype=tip, device=device), x_proba[:2])),
        'SAGEConv': sageconv_radi(tip),
    }
    log.info(str(tip) + ': ' + ', '.join(f'{k} {"radi" if v else "ne radi"}' for k, v in st.items()))
    if st['SAGEConv'] and RACUNSKI_TIP is None:
        RACUNSKI_TIP = tip
if RACUNSKI_TIP is None:
    RACUNSKI_TIP = torch.int64
log.info(f'najuzi tip za racunanje: {RACUNSKI_TIP}')


## Model i koraci treninga

In [ ]:
SKRIVENI, LR = 256, 0.01
KORAKA, SLOJEVA, OCENA_SVAKIH = 5000, 3, 250
UZORAKA_ZA_STRUKTURU = 20


class GraphSAGE(torch.nn.Module):
    def __init__(self, ulaz, skriveni, izlaz, slojeva, dropout=0.5):
        super().__init__()
        self.convs = torch.nn.ModuleList()
        self.bns = torch.nn.ModuleList()
        self.slojeva, self.dropout = slojeva, dropout
        self.convs.append(SAGEConv(ulaz, skriveni))
        self.bns.append(torch.nn.BatchNorm1d(skriveni))
        for _ in range(slojeva - 2):
            self.convs.append(SAGEConv(skriveni, skriveni))
            self.bns.append(torch.nn.BatchNorm1d(skriveni))
        self.convs.append(SAGEConv(skriveni, izlaz))

    def forward(self, x, edge_index):
        for i in range(self.slojeva - 1):
            x = self.convs[i](x, edge_index)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return self.convs[-1](x, edge_index)


def trening_korak(model, batch, opt):
    batch = batch.to(device, non_blocking=True)
    opt.zero_grad()
    out = model(batch.x, batch.edge_index)
    maska = batch.train_mask
    if maska.sum() == 0:
        return
    gubitak = F.cross_entropy(out[maska], batch.y[maska])
    gubitak.backward()
    opt.step()


def trening_korak_sastavljen(model, x, y, maska, ei, opt):
    opt.zero_grad()
    out = model(x, ei)
    if maska.sum() == 0:
        return
    gubitak = F.cross_entropy(out[maska], y[maska])
    gubitak.backward()
    opt.step()


## Ucitavanje grafa

In [ ]:
dataset = PygNodePropPredDataset(name='ogbn-arxiv', root='podaci')
data = dataset[0]
data.y = data.y.squeeze()
split_idx = dataset.get_idx_split()
evaluator = Evaluator(name='ogbn-arxiv')

data.edge_index = to_undirected(data.edge_index)
data.train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
data.train_mask[split_idx['train']] = True
log.info(f'ogbn-arxiv: {data.num_nodes:,} cvorova, {data.num_edges:,} potega')


@torch.no_grad()
def oceni(model):
    model.eval()
    x = data.x.to(device, non_blocking=True)
    ei = data.edge_index.to(device, non_blocking=True)
    out = model(x, ei).argmax(dim=-1, keepdim=True).cpu()
    del x, ei
    torch.cuda.empty_cache()
    return {s: evaluator.eval({'y_true': data.y[split_idx[s]].unsqueeze(1),
                               'y_pred': out[split_idx[s]]})['acc']
            for s in ['train', 'valid', 'test']}


## Particionisanje i priprema klastera

In [ ]:
def pripremi_klastere(K):
    t0 = time.time()
    cd = ClusterData(data, num_parts=K, recursive=False)
    metis_vreme = time.time() - t0
    klasteri = [kl for kl in ClusterLoader(cd, batch_size=1, shuffle=False)]
    najveci = max(kl.num_nodes for kl in klasteri)
    if najveci > INT16_MAKS:
        raise ValueError(f'najveci klaster {najveci} ne staje u int16 pri K={K}')
    return cd, klasteri, najveci, metis_vreme


## Prvi deo, trening u tri varijante po K i seedu

In [ ]:
for K in KLASTERI:
    cd, klasteri, najveci, metis_vreme = pripremi_klastere(K)
    potezi64 = [kl.edge_index for kl in klasteri]
    potezi16 = [p.to(torch.int16) for p in potezi64]
    velicine = [kl.num_nodes for kl in klasteri]
    mb64 = sum(p.numel() for p in potezi64) * 8 / 1e6
    mb16 = sum(p.numel() for p in potezi16) * 2 / 1e6
    mb_globalno = data.edge_index.numel() * 8 / 1e6
    log.info('-' * 70)
    log.info(f'K={K}: najveci klaster {najveci}, METIS {metis_vreme:.1f} s, struktura '
             f'globalno {mb_globalno:.1f} MB, izolovano int64 {mb64:.1f} MB, int16 {mb16:.1f} MB')

    for SEED in SEEDOVI:
        fajl = direktorijum(SEED) / 'int16.csv'
        uradjeno = vec_uradjeno(fajl, 'oznaka')
        for varijanta in VARIJANTE:
            oznaka = f'K{K}_{varijanta}'
            if oznaka in uradjeno:
                continue
            postavi_seed(SEED)
            model = GraphSAGE(data.num_features, SKRIVENI, dataset.num_classes, SLOJEVA).to(device)
            opt = torch.optim.Adam(model.parameters(), lr=LR)
            if varijanta == 'osnova':
                loader = ClusterLoader(cd, batch_size=KLASTERA_PO_BATCHU, shuffle=True,
                                       num_workers=8, persistent_workers=True)
                potezi, struktura = None, mb_globalno
            else:
                loader = None
                potezi = potezi64 if varijanta == 'izolovano_int64' else potezi16
                struktura = mb64 if varijanta == 'izolovano_int64' else mb16

            naj_val, naj_test, naj_korak, vrh = 0.0, float('nan'), 0, 0.0
            trening_vreme, korak, epoha = 0.0, 0, 0
            cvorova_l, potega_l = [], []
            k_cvorova, k_potega, k_zbir = 0, 0, 0
            torch.cuda.reset_peak_memory_stats()
            while korak < KORAKA:
                epoha += 1
                model.train()
                grupe = loader if varijanta == 'osnova' else torch.randperm(K).split(KLASTERA_PO_BATCHU)
                for grupa in grupe:
                    torch.cuda.synchronize()
                    t0 = time.time()
                    if varijanta == 'osnova':
                        n_cv, n_po = grupa.num_nodes, grupa.num_edges
                        zbir = int(grupa.edge_index.sum()) if korak < 10 else 0
                        trening_korak(model, grupa, opt)
                    else:
                        idx = grupa.tolist()
                        x = torch.cat([klasteri[j].x for j in idx]).to(device, non_blocking=True)
                        y = torch.cat([klasteri[j].y for j in idx]).to(device, non_blocking=True)
                        maska = torch.cat([klasteri[j].train_mask for j in idx]).to(device, non_blocking=True)
                        delovi = [potezi[j] for j in idx]
                        ei = torch.cat(delovi, dim=1).to(device, non_blocking=True).to(RACUNSKI_TIP)
                        brojevi = torch.tensor([p.size(1) for p in delovi], device=device)
                        of, ofs = 0, []
                        for j in idx:
                            ofs.append(of)
                            of += velicine[j]
                        ofseti = torch.tensor(ofs, dtype=RACUNSKI_TIP, device=device)
                        ei = ei + torch.repeat_interleave(ofseti, brojevi)
                        n_cv, n_po = x.size(0), ei.size(1)
                        zbir = int(ei.sum().item()) if korak < 10 else 0
                        trening_korak_sastavljen(model, x, y, maska, ei, opt)
                    torch.cuda.synchronize()
                    vrh = max(vrh, torch.cuda.max_memory_allocated() / 1e6)
                    trening_vreme += time.time() - t0
                    if korak < 10:
                        k_cvorova += n_cv
                        k_potega += n_po
                        k_zbir += zbir
                    if len(cvorova_l) < UZORAKA_ZA_STRUKTURU:
                        cvorova_l.append(n_cv)
                        potega_l.append(n_po)
                    korak += 1
                    if korak % OCENA_SVAKIH == 0:
                        a = oceni(model)
                        if a['valid'] > naj_val:
                            naj_val, naj_test, naj_korak = a['valid'], a['test'], korak
                        model.train()
                        torch.cuda.reset_peak_memory_stats()
                    if korak >= KORAKA:
                        break

            upisi_red(fajl, {
                'oznaka': oznaka, 'klastera': K, 'varijanta': varijanta,
                'tip_racunanja': str(RACUNSKI_TIP).replace('torch.', ''),
                'struktura_mb': struktura, 'najveci_klaster': najveci,
                'naj_val': naj_val, 'naj_test': naj_test, 'naj_korak': naj_korak,
                'epoha': epoha, 'metis_s': metis_vreme,
                'cvorova_po_koraku': float(np.mean(cvorova_l)),
                'potega_po_koraku': float(np.mean(potega_l)),
                'vreme_po_koraku_ms': trening_vreme / korak * 1000,
                'memorija_mb': vrh,
                'kontrola_cvorova': k_cvorova, 'kontrola_potega': k_potega,
                'kontrola_zbir': k_zbir})
            log.info(f'seed {SEED} | {oznaka}: test={naj_test*100:.2f}% (korak {naj_korak}), '
                     f'{trening_vreme/korak*1000:.1f} ms/korak, vrh {vrh:.0f} MB, '
                     f'struktura {struktura:.1f} MB')
            del model, opt
            if loader is not None:
                del loader
            gc.collect()
            torch.cuda.empty_cache()

    del cd, klasteri, potezi64, potezi16
    gc.collect()

log.info('Prvi deo zavrsen za sve seedove')


## Kontrola identiteta i sazetak

In [ ]:
problemi, razlike = [], []
for SEED in SEEDOVI:
    p = direktorijum(SEED) / 'int16.csv'
    if not p.exists():
        continue
    with open(p) as f:
        redovi = {r['oznaka']: r for r in csv.DictReader(f)}
    for K in KLASTERI:
        a = redovi.get(f'K{K}_izolovano_int64')
        b = redovi.get(f'K{K}_izolovano_int16')
        if a is None or b is None:
            continue
        for kol in ['kontrola_cvorova', 'kontrola_potega', 'kontrola_zbir']:
            if a[kol] != b[kol]:
                problemi.append((SEED, K, kol, a[kol], b[kol]))
        razlike.append(abs(float(a['naj_test']) - float(b['naj_test'])) * 100)

if problemi:
    log.info(f'NEUSPEH, sekvence batch-eva se razlikuju: {problemi}')
elif razlike:
    log.info(f'PROSLO, int64 i int16 varijanta imaju identicnu sekvencu batch-eva u svih {len(razlike)} poredjenja')
    log.info(f'razlika tacnosti izmedju int64 i int16: najveca {max(razlike):.3f} p.p., sme poticati samo od GPU nedeterminizma')

for K in KLASTERI:
    for varijanta in VARIJANTE:
        t, m, v = [], [], []
        st = None
        for SEED in SEEDOVI:
            p = direktorijum(SEED) / 'int16.csv'
            if not p.exists():
                continue
            with open(p) as f:
                redovi = {r['oznaka']: r for r in csv.DictReader(f)}
            r = redovi.get(f'K{K}_{varijanta}')
            if r is None:
                continue
            t.append(float(r['naj_test']) * 100)
            m.append(float(r['memorija_mb']))
            v.append(float(r['vreme_po_koraku_ms']))
            st = float(r['struktura_mb'])
        if len(t) > 1:
            log.info(f'K={K} {varijanta:16s}: tacnost {np.mean(t):.2f} +/- {np.std(t, ddof=1):.2f} %, '
                     f'vrh memorije {np.mean(m):.0f} MB, {np.mean(v):.1f} ms/korak, '
                     f'struktura {st:.1f} MB')


## Drugi deo, skladiste i prenos strukture na sva cetiri grafa

Bez treninga. Za svaki graf se izmeri velicina strukture u int64 i int16, velicina mape
globalnih indeksa u int64 i int32, i vreme prenosa strukture na karticu u oba tipa,
najbolje od pet ponavljanja. Uz to se belezi najveci klaster i koliko bi klastera
trazio int8, sto pokazuje zasto je int8 nepraktican.


In [ ]:
GRAFOVI = [('ogbn-arxiv', 100), ('Reddit', 100), ('Yelp', 100), ('ogbn-products', 500)]
SKL = REZ / 'skladiste.csv'


def ucitaj(ime):
    if ime.startswith('ogbn'):
        ds = PygNodePropPredDataset(name=ime, root='podaci')
        d = ds[0]
        if ime == 'ogbn-arxiv':
            d.edge_index = to_undirected(d.edge_index)
        return d
    if ime == 'Reddit':
        from torch_geometric.datasets import Reddit
        return Reddit(root='podaci/Reddit')[0]
    from torch_geometric.datasets import Yelp
    return Yelp(root='podaci/Yelp')[0]


def prenos_ms(t, ponavljanja=5):
    vremena = []
    for _ in range(ponavljanja):
        torch.cuda.synchronize()
        t0 = time.time()
        g = t.to(device)
        torch.cuda.synchronize()
        vremena.append(time.time() - t0)
        del g
        torch.cuda.empty_cache()
    return min(vremena) * 1000


gotovo = vec_uradjeno(SKL, 'graf')
for ime, K in GRAFOVI:
    if ime in gotovo:
        continue
    log.info('-' * 70)
    log.info(f'{ime}: ucitavanje i particionisanje na K={K}')
    d = ucitaj(ime)
    t0 = time.time()
    cd = ClusterData(d, num_parts=K, recursive=False)
    metis_s = time.time() - t0
    velicine, delovi = [], []
    for kl in ClusterLoader(cd, batch_size=1, shuffle=False):
        velicine.append(kl.num_nodes)
        delovi.append(kl.edge_index)
    najveci = max(velicine)
    ei64 = torch.cat(delovi, dim=1).contiguous()
    del delovi
    staje = najveci <= INT16_MAKS
    red = {'graf': ime, 'K': K, 'cvorova': d.num_nodes, 'potega_ukupno': d.num_edges,
           'potega_zadrzano': ei64.size(1),
           'udeo_zadrzanih_pct': ei64.size(1) / d.num_edges * 100,
           'najveci_klaster': najveci, 'staje_int16': int(staje), 'metis_s': metis_s,
           'struktura_int64_mb': ei64.numel() * 8 / 1e6,
           'struktura_int16_mb': ei64.numel() * 2 / 1e6,
           'mapa_int64_mb': d.num_nodes * 8 / 1e6,
           'mapa_int32_mb': d.num_nodes * 4 / 1e6,
           'najmanje_klastera_za_int8': int(np.ceil(d.num_nodes / 127)),
           'prenos_int64_ms': prenos_ms(ei64),
           'prenos_int16_ms': prenos_ms(ei64.to(torch.int16)) if staje else float('nan')}
    upisi_red(SKL, red)
    log.info(f'{ime}: najveci klaster {najveci}, staje u int16: {staje}, struktura '
             f'{red["struktura_int64_mb"]:.0f} naspram {red["struktura_int16_mb"]:.0f} MB, '
             f'prenos {red["prenos_int64_ms"]:.1f} naspram {red["prenos_int16_ms"]:.1f} ms')
    del d, cd, ei64, velicine
    gc.collect()
    torch.cuda.empty_cache()

log.info('Drugi deo zavrsen')


## Arhiviranje rezultata

In [ ]:
import zipfile

arhiva = Path('rezultati_int16.zip')
with zipfile.ZipFile(arhiva, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(REZ.rglob('*')):
        if p.is_file():
            z.write(p)
log.info(f'{arhiva}: {arhiva.stat().st_size / 1e6:.1f} MB, preuzeti i sacuvati u projekat')
